# 07 – Person Alternative Names

Esplorazione e data cleaning del dataset `person_alternate_names.csv`.

**Colonne:**
| Colonna | Descrizione |
|---|---|
| `person_mal_id` | ID univoco della persona su MyAnimeList |
| `alt_name` | Nome alternativo della persona |

## 1. Import e caricamento dati
Importiamo le librerie necessarie e carichiamo il file csv. Facciamo una esplorazione generica per capire la struttura e le caratteristiche del dataset.

In [1]:
import pandas as pd
import numpy as np
from dataset_analyzer import analyze
from foreign_key_analyzer import check_fk

df_alt = pd.read_csv('../datasets/person_alternate_names.csv')
print(f'Shape: {df_alt.shape}')
print()
df_alt.info()
print()
df_alt.head()

Shape: (20465, 2)

<class 'pandas.DataFrame'>
RangeIndex: 20465 entries, 0 to 20464
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   person_mal_id  20465 non-null  int64
 1   alt_name       20447 non-null  str  
dtypes: int64(1), str(1)
memory usage: 319.9 KB



,person_mal_id,alt_name
0,1,Seki Mondoya
1,1,門戸 開
2,1,Monto Hiraku
3,3,雪野五月
4,10,Kevin Hatcher


**Osservazioni iniziali:**
- Il dataset contiene **20.465 righe** e **2 colonne**.
- `person_mal_id` è completo: **nessun valore nullo**.
- `alt_name` presenta **18 valori nulli** (20.447 non-null su 20.465), da gestire in fase di cleaning.
- I tipi di dati sono adeguati: `int64` e `str`.

## 1.1 Rimozione duplicati esatti

Prima dell'analisi per colonna, rimuoviamo le righe con valori identici in **tutte** le colonne, mantenendo solo la prima occorrenza.

In [2]:
n_originale = len(df_alt)

mask_dup = df_alt.duplicated(keep=False)
n_righe_coinvolte = mask_dup.sum()
n_gruppi = df_alt[mask_dup].duplicated(keep='first').sum()
n_tenute = n_righe_coinvolte - n_gruppi

print(f'Righe totali coinvolte in duplicazioni : {n_righe_coinvolte:,}')
print(f'  → prime occorrenze mantenute         : {n_tenute:,}')
print(f'  → occorrenze extra rimosse           : {n_gruppi:,}')
print()

df_alt.drop_duplicates(keep='first', inplace=True)
print(f'Righe prima della rimozione : {n_originale:,}')
print(f'Righe dopo la rimozione     : {len(df_alt):,}')

Righe totali coinvolte in duplicazioni : 68
  → prime occorrenze mantenute         : 29
  → occorrenze extra rimosse           : 39

Righe prima della rimozione : 20,465
Righe dopo la rimozione     : 20,426


**Osservazioni:**

**68 righe** (0.33% del totale) risultano coinvolte in duplicazioni esatte su entrambe le colonne. Di queste, **29** sono prime occorrenze mantenute e **39** occorrenze extra rimosse. Dopo la rimozione il dataset scende da 20.465 a **20.426 righe**.

Adesso che siamo sicuri che tutte le righe sono uniche, iniziamo l'analisi per colonne utilizzando la nostra libreria `dataset_analyzer`.

## 2. Analisi colonna per colonna

### 2.1 `person_mal_id`

Questa colonna è una **chiave esterna** che referenzia la chiave primaria di `person_details.csv`.

I valori duplicati sono **attesi**: la stessa persona può avere più nomi alternativi.

I controlli rilevanti sono:
- **Valori nulli**: una chiave esterna nulla indica una riga senza riferimento che va rimossa.
- **Integrità referenziale**: ogni ID presente qui deve esistere in `person_details_clean.csv`.

Usiamo quindi `check_fk` al posto di `analyze`, che effettua entrambi i controlli.

In [3]:
df_persons = pd.read_csv('../datasets_cleaned/person_details_clean.csv')

mask_orphan = check_fk(df_alt['person_mal_id'], df_persons['person_mal_id'], child_df=df_alt
)

print(f'Null in person_mal_id               : {df_alt["person_mal_id"].isna().sum()}')
print(f'Duplicati in person_mal_id (attesi) : {df_alt["person_mal_id"].duplicated().sum():,}')


  Colonna FK  (tabella figlia)        person_mal_id
  Colonna PK  (tabella padre)         person_mal_id
────────────────────────────────────────────────────────────────────────────────

Riepilogo conteggi
────────────────────────────────────────────────────────────────────────────────

  Righe totali (tabella figlia)       20,426
  Valori null nella FK                0  (0.00%)
  Valori non-null nella FK            20,426  (100.00%)
  Valori unici nella PK padre         75,211

  ✓  Righe con FK valida              20,426  (100.00%)
  ✗  Righe orfane (FK non in PK)      0  (0.00%)
     → ID orfani unici                0

Valutazione integrità referenziale
────────────────────────────────────────────────────────────────────────────────

Nessuna violazione. Tutti i valori FK esistono nella tabella padre.

Null in person_mal_id               : 0
Duplicati in person_mal_id (attesi) : 8,050


**Osservazioni:**
- **Nessun valore nullo**: tutti i record hanno un ID di riferimento valido.
- **Integrità referenziale**: non ci sono righe orfane.

Nessuna pulizia è necessaria.

### 2.2 `alt_name`

È l'unica colonna con contenuto semantico del dataset: contiene il nome alternativo della persona in forma testuale libera.

A differenza di `person_mal_id`, qui ci interessa capire la distribuzione delle lunghezze, la presenza di valori nulli, duplicati, caratteri anomali, etc. Per questo utilizziamo `analyze`.

I duplicati sono **attesi** in quanto nomi generici possono comparire per persone diverse e non costituiscono un problema.

In [4]:
df_alt['alt_name'] = df_alt['alt_name'].str.strip()
analyze(df_alt['alt_name'])


  Nome serie                     alt_name
  dtype                          str
  Dimensione                     20,426
  Range indice                   0 … 20464
  Tipo                           STRINGA / OGGETTO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   20,426
  Valori non nulli               20,408  (99.91%)
  Null / NaN                     18  (0.09%)
    di cui stringhe vuote        0  (convertite a NULL)
  Valori unici                   20,248  (99.13%)
  Valori duplicati               160  (0.78%)

Analisi lunghezza stringhe
────────────────────────────────────────────────────────────────────────────────

  Lunghezza min                  1  caratteri
  Lunghezza max                  58  caratteri
  Lunghezza media                9.63  caratteri
  Lunghezza mediana              9.0000  caratteri
  Dev. standard lunghezza        5.69
  IQR lunghezza                  8.0000

Valori di lunghez

**Osservazioni:**

- Ci sono 160 duplicati che sono attesi in quanto diverse persone possono condividere lo stesso alias. In più, dai valori più frequenti si nota che i duplicati consistono principalmente in nomi di gruppi musicali o nomi di studio. 
- Ci sono 8 alias contenenti URL che sarebbe anomalo. Stampiamo per verificare se rimuovere o meno. 
- Ci sono 11 alias contenenti `@` e 15 alias solo numerici. Stampiamo per verificare se si tratta di anomalie.

In [5]:
mask_at      = df_alt['alt_name'].str.contains('@', regex=False, na=False)
mask_url     = df_alt['alt_name'].str.contains(r'https?://', regex=True, na=False)
mask_numeric = df_alt['alt_name'].str.fullmatch(r'\d+', na=False)

print(f"Alias con '@'          : {mask_at.sum()}")
print(df_alt[mask_at][['person_mal_id', 'alt_name']].to_string(index=False))
print()

print(f"Alias con URL          : {mask_url.sum()}")
print(df_alt[mask_url][['person_mal_id', 'alt_name']].to_string(index=False))
print()

print(f"Alias solo numerici    : {mask_numeric.sum()}")
print(df_alt[mask_numeric][['person_mal_id', 'alt_name']].to_string(index=False))

Alias con '@'          : 11
 person_mal_id      alt_name
         20322 Joukai@Outori
         24815           唯@ｗ
         26795       Rip@Lip
         36922       Akiha @
         39963 cosMo@BousouP
         41198       ft@pepo
         52401          CO2@
         68452        Arufa@
         68452         あるふぁ@
         74789          CH@R
         84032         Kavk@

Alias con URL          : 8
 person_mal_id                         alt_name
         38508           http://www.suoyon.net/
         47280        https://amazatosugar.com/
         51560       https://www.tarapammi.com/
         52285 https://tristone.co.jp/kiyozuka/
         55482         http://tetubin.ninpou.jp
         80967     http://www.johnhasler.co.uk/
         82243       https://tooncracker.co.jp/
         83688        https://yokomizoyuri.com/

Alias solo numerici    : 15
 person_mal_id alt_name
         11178      000
         23551      415
         49452     6109
         49573     3285
         49650 

**Pulizie necessarie:**
- 18 null residui → rimuovere le righe (una riga senza nome alternativo non ha significato)
- 8 alias contenenti URL → rimuovere le righe (un URL non è un nome alternativo)
- I nomi duplicati sono attesi: lo stesso nome può comparire per persone diverse

In [6]:
mask_url = df_alt['alt_name'].str.contains(r'https?://', regex=True, na=False)
print(f'Righe con URL prima della pulizia   : {mask_url.sum()}')
df_alt = df_alt[~mask_url].reset_index(drop=True)
print(f'Righe dopo rimozione URL            : {len(df_alt):,}')
print()

print(f'Null in alt_name prima della pulizia: {df_alt["alt_name"].isna().sum()}')
df_alt.dropna(subset=['alt_name'], inplace=True)
print(f'Null in alt_name dopo la pulizia    : {df_alt["alt_name"].isna().sum()}')
print(f'Righe dopo pulizia alt_name         : {len(df_alt):,}')

Righe con URL prima della pulizia   : 8
Righe dopo rimozione URL            : 20,418

Null in alt_name prima della pulizia: 18
Null in alt_name dopo la pulizia    : 0
Righe dopo pulizia alt_name         : 20,400


### 2.3 Chiave primaria `(person_mal_id, alt_name)`

La chiave primaria naturale è la coppia `(person_mal_id, alt_name)`: verifichiamo che sia univoca dopo la pulizia.

In [7]:
n_pk_dup = df_alt.duplicated(subset=['person_mal_id', 'alt_name'], keep=False).sum()
print(f'Duplicati su (person_mal_id, alt_name): {n_pk_dup}')

if n_pk_dup > 0:
    print('→ Rimozione duplicati sulla chiave primaria...')
    df_alt.drop_duplicates(subset=['person_mal_id', 'alt_name'], keep='first', inplace=True)
    print(f'Righe dopo la rimozione: {len(df_alt):,}')
else:
    print('→ Chiave primaria già univoca, nessuna operazione richiesta.')

Duplicati su (person_mal_id, alt_name): 0
→ Chiave primaria già univoca, nessuna operazione richiesta.


## 3. Riepilogo e Salvataggio
Le operazioni di pulizia sono state effettuate colonna per colonna nella sezione 2. In questa sezione riepiloghiamo il risultato ed effettuiamo il salvataggio del dataset finale.

In [8]:
print('Riepilogo Dataset Pulito')
print(f'Righe originali      : {n_originale:>10,}')
print(f'Righe dopo cleaning  : {len(df_alt):>10,}')
print(f'Righe rimosse totali : {n_originale - len(df_alt):>10,}')
print()
df_alt.to_csv('../datasets_cleaned/person_alternate_names_clean.csv', index=False)
print('Salvato: datasets_cleaned/person_alternate_names_clean.csv')

Riepilogo Dataset Pulito
Righe originali      :     20,465
Righe dopo cleaning  :     20,400
Righe rimosse totali :         65

Salvato: datasets_cleaned/person_alternate_names_clean.csv
